# Probabilistic Verification

This notebook focuses on evaluating probabilistic forecasts, typically from ensemble prediction systems.
Probabilistic metrics assess the reliability and resolution of the forecast distribution.

## Key Metrics

*   **CRPS (Continuous Ranked Probability Score)**: A proper scoring rule that generalizes MAE to probabilistic forecasts. It measures the distance between the forecast cumulative distribution function (CDF) and the empirical CDF of the observation.
*   **PIT (Probability Integral Transform)**: Checks for calibration (reliability). A perfectly calibrated ensemble should have a uniform PIT histogram.
    *   U-shape: Under-dispersive (overconfident).
    *   A-shape: Over-dispersive (underconfident).
*   **Spread-Skill Ratio**: Compares the ensemble spread (standard deviation) to the RMSE of the ensemble mean. Ideally, spread should match skill (ratio = 1).

## Relevant Code

The core logic resides in `src/swissclim_evaluations/metrics/probabilistic.py`.

Key functions:
*   [`run_probabilistic`](../src/swissclim_evaluations/metrics/probabilistic.py#L238): Orchestrates the calculation of CRPS and PIT.
*   [`probability_integral_transform`](../src/swissclim_evaluations/metrics/probabilistic.py#L142): Computes the PIT values.
*   [`compute_wbx_crps`](../src/swissclim_evaluations/metrics/probabilistic.py#L84): Wraps WeatherBenchX CRPS calculation.

## Configuration

Please specify the path to your configuration file below.

In [ ]:
# Define the path to the configuration file
# If set to None, the notebook will attempt to auto-discover 'config/full_verification_run_prob.yaml' or 'config/example_config.yaml'
config_path_str = "../config/full_verification_run_prob.yaml"

In [ ]:
from pathlib import Path

import pandas as pd
import xarray as xr
from IPython.display import Image, display

from swissclim_evaluations.cli import _load_yaml, prepare_datasets, run_selected
from swissclim_evaluations.metrics.probabilistic import plot_probabilistic, run_probabilistic_wbx

In [ ]:
from swissclim_evaluations.helpers import display_outputs

In [ ]:
# Locate configuration relative to the notebook location
cfg_path = None

if config_path_str:
    candidate = Path(config_path_str)
    if not candidate.is_absolute():
        candidate = (Path.cwd() / candidate).resolve()
    if candidate.is_file():
        cfg_path = candidate
        print(f"Using configuration: {cfg_path}")
    else:
        print(f"Warning: Config file not found at {candidate}. Falling back to auto-discovery.")

if cfg_path is None:
    # Try to find full_verification_run_prob.yaml first, then fall back to example_config.yaml
    config_names = ["full_verification_run_prob.yaml", "example_config.yaml"]

    for name in config_names:
        for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
            candidate = base / "config" / name
            if candidate.is_file():
                cfg_path = candidate
                print(f"Using configuration: {cfg_path}")
                break
        if cfg_path:
            break

if cfg_path is None:
    raise FileNotFoundError(
        "Could not find config file (full_verification_run_prob.yaml or example_config.yaml) in cwd, parent, or grandparent directories."
    )

# Load configuration via project helper to keep parity with CLI
cfg = _load_yaml(cfg_path)

# Prepare datasets using the CLI pipeline (handles selection, alignment, ensemble policy)
ds_target, ds_prediction, ds_std, ds_prediction_std = prepare_datasets(cfg)

In [ ]:
# Show the prepared datasets (quick peek)
display(ds_target)
display(ds_prediction)

In [ ]:
# Ensure the config has probabilistic enabled
cfg_modules = cfg.get("modules", {})
if not cfg_modules.get("probabilistic", False):
    print("Enabling modules.probabilistic in-memory for this run…")
    cfg_modules["probabilistic"] = True
    cfg["modules"] = cfg_modules

# Set up output directory from config relative to project root (parent of notebooks)
project_root = Path.cwd().parent
cfg_output = cfg.get("paths", {}).get("output_root", "output/notebook_prob")
out_root = (
    (project_root / cfg_output).resolve()
    if not cfg_output.startswith("/")
    else Path(cfg_output)
)
out_root.mkdir(parents=True, exist_ok=True)
print(f"Output root: {out_root}")

# Optional: Execute the full pipeline (data prep, metrics, plots) using CLI logic.
# This will generate all configured artifacts to out_root respecting cfg['modules'] flags
# and cfg['plotting']['output_mode'] (e.g., 'npz' to export numeric data only).
RUN_ALL = False  # Set to True to run everything end-to-end automatically.

if RUN_ALL:
    run_selected(cfg)
    print(f"All selected modules executed. Outputs under: {out_root}")

## CRPS Maps
Continuous Ranked Probability Score (CRPS) maps.
Shows the spatial distribution of the CRPS metric. Lower is better.

**Relevant Code:**
*   [`run_probabilistic`](../src/swissclim_evaluations/metrics/probabilistic.py#L238): Computes CRPS summary and fields.
*   [`plot_probabilistic`](../src/swissclim_evaluations/metrics/probabilistic.py#L661): Generates the CRPS map plots.

In [ ]:
# Display CRPS Maps
print("Displaying CRPS Maps...")
display_outputs(out_root / "probabilistic", pattern_img="crps_map_*.png", pattern_csv="crps_summary*.csv", exclude_pattern="grid")

## PIT Histograms
Probability Integral Transform (PIT) histograms.
Checks for calibration. A flat histogram indicates a well-calibrated ensemble.
U-shape indicates under-dispersion (overconfident).
Inverted U-shape indicates over-dispersion (underconfident).

**Relevant Code:**
*   [`probability_integral_transform`](../src/swissclim_evaluations/metrics/probabilistic.py#L142): Computes the PIT values.
*   [`plot_probabilistic`](../src/swissclim_evaluations/metrics/probabilistic.py#L661): Generates the PIT histogram plots.

In [ ]:
# Display PIT Histograms
print("Displaying PIT Histograms...")
display_outputs(out_root / "probabilistic", pattern_img="pit_hist_*.png", pattern_csv="None", exclude_pattern="grid")

## Spread-Error Diagrams
Compares the ensemble spread (standard deviation) with the RMSE of the ensemble mean.
For a well-calibrated ensemble, spread should match the error.

**Relevant Code:**
*   [`run_probabilistic`](../src/swissclim_evaluations/metrics/probabilistic.py#L238): Calculates Spread-Skill Ratio.
*   [`plot_probabilistic`](../src/swissclim_evaluations/metrics/probabilistic.py#L661): Generates the Spread-Error diagrams.

In [ ]:
# Display SSR Spatial Plots (Spread-Skill Ratio by Region)
print("Displaying SSR Spatial Plots...")
display_outputs(out_root / "probabilistic", pattern_img="wbx_spatial_*.png", pattern_csv="spread_skill*.csv")